In [ ]:
import pandas as pd
data = pd.read_csv('/content/combined_mid_large.csv')
data.head()

,Date,symbol,Open,High,Low,Close,Volume,Stock Name,Daily Return,Lagged Return,...,MACD_Signal,MACD_Histogram,Volatility,OBV,Bollinger_Middle,Bollinger_Upper,Bollinger_Lower,Momentum,ATR,ADX
0,1999-10-20 03:45:00,NSE:HDFCBANK,10.500,10.695,10.225,10.350,1192470.0,HDFC Bank Ltd.,0.001936,10.330,...,0.237705,0.022557,0.281216,85496890.0,9.81500,10.401345,9.228655,0.740,0.555694,18.616982
1,1999-10-21 03:45:00,NSE:HDFCBANK,10.495,10.495,9.800,9.835,1333700.0,HDFC Bank Ltd.,-0.049758,10.350,...,0.235913,-0.007168,0.268584,84163190.0,9.82950,10.402613,9.256387,0.305,0.565645,17.657119
2,1999-10-22 03:45:00,NSE:HDFCBANK,9.900,10.035,9.800,9.995,2888020.0,HDFC Bank Ltd.,0.016268,9.835,...,0.231572,-0.017362,0.269235,87051210.0,9.85325,10.412287,9.294213,0.235,0.542027,16.765817
3,1999-10-25 03:45:00,NSE:HDFCBANK,10.290,10.300,9.500,9.615,1667150.0,HDFC Bank Ltd.,-0.038019,9.995,...,0.219271,-0.049204,0.257005,85384060.0,9.85550,10.410305,9.300695,-0.300,0.560454,16.092559
4,1999-10-26 03:45:00,NSE:HDFCBANK,9.450,9.700,9.400,9.440,1467050.0,HDFC Bank Ltd.,-0.018201,9.615,...,0.199334,-0.079750,0.276340,83917010.0,9.85250,10.415550,9.289450,-0.445,0.541850,15.738599


### Regression: PART!

In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
os.makedirs('/mnt/data', exist_ok=True)
# Load dataset
data = pd.read_csv('/content/combined_mid_large.csv')

# Convert 'Date' to datetime
data['Date'] = pd.to_datetime(data['Date'])

# Filter data to include only from 2010 onward
data = data[data['Date'] >= '2010-01-01']

# Store the results in a dictionary to pass to Section 2
model_results = {'stock': [], 'RMSE': [], 'MAE': [], 'R2': [], 'Model': []}

# Process each stock separately
stocks = data['symbol'].unique()

for stock in stocks:
    stock_data = data[data['symbol'] == stock].copy()

    # Calculate daily returns
    stock_data['Daily_Return'] = stock_data['Close'].pct_change()
    stock_data.dropna(inplace=True)

    # Prepare features and target
    features = ['SMA_9', 'SMA_21', 'EMA_9', 'EMA_21', 'RSI', 'MACD', 'Volatility', 'Momentum', 'ATR', 'ADX']
    X = stock_data[features]
    y = stock_data['Daily_Return']

    # Split data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    # Scale features for ElasticNet & LGBM (RandomForest & XGBoost do not require scaling)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define models and their hyperparameters for GridSearch
    models = {
        'ElasticNet': (ElasticNet(), {'alpha': [0.01, 0.1, 1], 'l1_ratio': [0.1, 0.5, 0.9]}),
        'RandomForest': (RandomForestRegressor(), {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}),
        'XGBoost': (XGBRegressor(), {'learning_rate': [0.01, 0.1], 'n_estimators': [100, 200]}),
        'LightGBM': (LGBMRegressor(), {'learning_rate': [0.01, 0.1], 'n_estimators': [100, 200]})
    }

    # Perform GridSearch and store the best results
    for model_name, (model, params) in models.items():
        if model_name in ['ElasticNet']:
            X_train_model, X_test_model = X_train_scaled, X_test_scaled
        else:
            X_train_model, X_test_model = X_train, X_test

        grid_search = GridSearchCV(model, params, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
        grid_search.fit(X_train_model, y_train)

        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test_model)

        rmse = mean_squared_error(y_test, y_pred, squared=False)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Store results for portfolio optimization
        model_results['stock'].append(stock)
        model_results['RMSE'].append(rmse)
        model_results['MAE'].append(mae)
        model_results['R2'].append(r2)
        model_results['Model'].append(model_name)

# Save results to DataFrame
results_df = pd.DataFrame(model_results)
results_df.to_csv('/mnt/data/regression_results.csv', index=False)


/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To 

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000399 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 2948, number of used features: 10
[LightGBM] [Info] Start training from score 0.000849


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'roo

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000542 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 2948, number of used features: 10
[LightGBM] [Info] Start training from score 0.000737


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'roo

...

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'roo

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000772 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 3945, number of used features: 10
[LightGBM] [Info] Start training from score 0.000568


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'roo

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000199 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 1736, number of used features: 10
[LightGBM] [Info] Start training from score 0.000779


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [ ]:
from google.colab import files
files.download('/mnt/data/regression_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from google.colab import files

# Ensure the output directory exists
os.makedirs('/mnt/data', exist_ok=True)

# Load dataset
data = pd.read_csv('/content/combined_mid_large.csv')

# Convert 'Date' to datetime
data['Date'] = pd.to_datetime(data['Date'])

# Filter data to include only from 2010 onward
data = data[data['Date'] >= '2010-01-01']

# Store the results in a dictionary
model_results = {'stock': [], 'RMSE': [], 'MAE': [], 'R2': [], 'Model': []}
best_model_per_stock = {'stock': [], 'Best_Model': [], 'Best_RMSE': [], 'Best_MAE': [], 'Best_R2': []}

# Process each stock separately
stocks = data['symbol'].unique()

for stock in stocks:
    stock_data = data[data['symbol'] == stock].copy()

    # Calculate daily returns
    stock_data['Daily_Return'] = stock_data['Close'].pct_change()
    stock_data.dropna(inplace=True)

    # Prepare features and target
    features = ['SMA_9', 'SMA_21', 'EMA_9', 'EMA_21', 'RSI', 'MACD', 'Volatility', 'Momentum', 'ATR', 'ADX']
    X = stock_data[features]
    y = stock_data['Daily_Return']

    # Split data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    # Scale features for ElasticNet & LGBM (RandomForest & XGBoost do not require scaling)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define models and their hyperparameters for GridSearch
    models = {
        'ElasticNet': (ElasticNet(), {'alpha': [0.01, 0.1, 1], 'l1_ratio': [0.1, 0.5, 0.9]}),
        'RandomForest': (RandomForestRegressor(), {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}),
        'XGBoost': (XGBRegressor(), {'learning_rate': [0.01, 0.1], 'n_estimators': [100, 200]}),
        'LightGBM': (LGBMRegressor(), {'learning_rate': [0.01, 0.1], 'n_estimators': [100, 200]})
    }

    # Store the best model for the current stock
    best_model_name = None
    best_rmse = float('inf')
    best_mae = None
    best_r2 = None

    # Perform GridSearch and store the best results
    for model_name, (model, params) in models.items():
        if model_name in ['ElasticNet']:
            X_train_model, X_test_model = X_train_scaled, X_test_scaled
        else:
            X_train_model, X_test_model = X_train, X_test

        grid_search = GridSearchCV(model, params, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
        grid_search.fit(X_train_model, y_train)

        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test_model)

        rmse = mean_squared_error(y_test, y_pred, squared=False)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Store results for overall tracking
        model_results['stock'].append(stock)
        model_results['RMSE'].append(rmse)
        model_results['MAE'].append(mae)
        model_results['R2'].append(r2)
        model_results['Model'].append(model_name)

        # Update best model for the current stock
        if rmse < best_rmse:
            best_model_name = model_name
            best_rmse = rmse
            best_mae = mae
            best_r2 = r2

    # Store the best model for this stock
    best_model_per_stock['stock'].append(stock)
    best_model_per_stock['Best_Model'].append(best_model_name)
    best_model_per_stock['Best_RMSE'].append(best_rmse)
    best_model_per_stock['Best_MAE'].append(best_mae)
    best_model_per_stock['Best_R2'].append(best_r2)

# Save results to DataFrames
results_df = pd.DataFrame(model_results)
best_models_df = pd.DataFrame(best_model_per_stock)

# Save to CSV
results_df.to_csv('/mnt/data/regression_results.csv', index=False)
best_models_df.to_csv('/mnt/data/best_models_per_stock.csv', index=False)

# Download best models CSV
files.download('/mnt/data/best_models_per_stock.csv')


/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To 

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000648 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 2948, number of used features: 10
[LightGBM] [Info] Start training from score 0.000849


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'roo

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000320 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 2948, number of used features: 10
[LightGBM] [Info] Start training from score 0.000737


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'roo

...

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'roo

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000324 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 2948, number of used features: 10
[LightGBM] [Info] Start training from score 0.000410


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Optimization and Best portfolio selection section 2

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import plotly.express as px
import plotly.graph_objects as go

# Load regression results and stock data
regression_results = pd.read_csv('/mnt/data/regression_results.csv')  # Regression results from section 1
data = pd.read_csv('/content/combined_mid_large.csv')  # Stock data
data['Date'] = pd.to_datetime(data['Date'])

# Calculate daily returns for each stock
data['Daily_Return'] = data.groupby('symbol')['Close'].pct_change()
returns_df = data.pivot_table(values='Daily_Return', index='Date', columns='symbol').dropna()

# Log returns (for stability)
log_returns_df = np.log(1 + returns_df)
mean_returns = log_returns_df.mean() * 252  # Annualized
cov_matrix = log_returns_df.cov() * 252  # Annualized

# Risk-free rate
risk_free_rate = 0.03

# Portfolio Optimization Function
def portfolio_metrics(weights):
    portfolio_return = np.sum(weights * mean_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
    return portfolio_return, portfolio_volatility, sharpe_ratio

# Function to minimize negative Sharpe ratio
def negative_sharpe(weights):
    _, _, sharpe_ratio = portfolio_metrics(weights)
    return -sharpe_ratio

# Initial weights, bounds, and constraints
num_assets = len(mean_returns)
initial_weights = np.ones(num_assets) / num_assets
bounds = [(0, 1) for _ in range(num_assets)]
constraints = [{'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1}]

# Optimization
result = minimize(negative_sharpe, initial_weights, bounds=bounds, constraints=constraints)

# Optimal weights
optimal_weights = result.x
optimal_return, optimal_volatility, optimal_sharpe = portfolio_metrics(optimal_weights)

# Save best portfolio details
best_portfolio = pd.DataFrame({
    'Stock': mean_returns.index,
    'Weight': optimal_weights,
    'Expected Return': mean_returns.values,
    'Volatility': np.sqrt(np.diag(cov_matrix)),
})
best_portfolio['Risk'] = best_portfolio['Weight'] * best_portfolio['Volatility']

# Calculate portfolio returns over time (cumulative)
portfolio_returns = (returns_df * optimal_weights).sum(axis=1)
cumulative_portfolio_returns = (1 + portfolio_returns).cumprod()

# Save portfolio results
best_portfolio.to_csv('/mnt/data/best_portfolio.csv', index=False)
cumulative_portfolio_returns.to_csv('/mnt/data/cumulative_returns.csv', header=['Portfolio Value'])

# Save all portfolios tested
all_portfolios = []

# Grid search across weights for reporting purposes
for _ in range(500):  # Randomized search over portfolios
    random_weights = np.random.dirichlet(np.ones(num_assets))
    ret, vol, sharpe = portfolio_metrics(random_weights)
    all_portfolios.append({
        'Weights': random_weights,
        'Return': ret,
        'Volatility': vol,
        'Sharpe Ratio': sharpe
    })

all_portfolios_df = pd.DataFrame(all_portfolios)
all_portfolios_df.to_csv('/mnt/data/all_portfolios.csv', index=False)

# Visualization 1: Line Graph of Portfolio Value Over Time
fig1 = px.line(
    x=cumulative_portfolio_returns.index,
    y=cumulative_portfolio_returns.values,
    title='Cumulative Portfolio Returns (2010 - Present)',
    labels={'x': 'Date', 'y': 'Portfolio Value (₹)'}
)
fig1.show()

# Visualization 2: Bar Graph of Yearly Portfolio Returns
import plotly.express as px

# Filter annual_returns to include only years from 2010 to the present
annual_returns_filtered = annual_returns[annual_returns.index >= 2010]

# Create the bar chart for filtered annual returns
fig2 = px.bar(
    x=annual_returns_filtered.index,
    y=annual_returns_filtered.values * 100,
    title='Yearly Portfolio Returns (%) (2010 - Present)',
    labels={'x': 'Year', 'y': 'Annual Return (%)'}
)

# Update layout for better visualization
fig2.update_layout(
    xaxis_title='Year',
    yaxis_title='Return (%)',
    xaxis=dict(tickmode='linear'),  # Ensure all years are shown as ticks
    template='plotly_white'
)

# Show the plot
fig2.show()


# Visualization 3: Plot Showing Different Portfolios and Best One
fig3 = px.scatter(
    all_portfolios_df,
    x='Volatility',
    y='Return',
    color='Sharpe Ratio',
    title='Portfolios Tested (Volatility vs. Return)',
    labels={'Volatility': 'Risk (Volatility)', 'Return': 'Reward (Return)'},
    hover_data=['Sharpe Ratio']
)
fig3.add_trace(
    go.Scatter(
        x=[optimal_volatility],
        y=[optimal_return],
        mode='markers',
        marker=dict(color='red', size=12),
        name='Best Portfolio'
    )
)
fig3.show()

# Visualization 4: Bar Graph of Portfolio Weights
fig4 = px.bar(
    best_portfolio,
    x='Stock',
    y='Weight',
    title='Best Portfolio Weights',
    labels={'Weight': 'Allocation'},
    text='Weight'
)
fig4.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig4.show()


In [ ]:
import plotly.express as px

# Filter annual_returns to include only years from 2010 to the present
annual_returns_filtered = annual_returns[annual_returns.index >= 2010]

# Create the bar chart for filtered annual returns
fig2 = px.bar(
    x=annual_returns_filtered.index,
    y=annual_returns_filtered.values * 100,
    title='Yearly Portfolio Returns (%) (2010 - Present)',
    labels={'x': 'Year', 'y': 'Annual Return (%)'},
    color=annual_returns_filtered.values * 100,  # Use color for intensity
    color_continuous_scale='Teal'  # Elegant, presentation-friendly color palette
)

# Update layout for better visualization
fig2.update_layout(
    xaxis_title='Year',
    yaxis_title='Return (%)',
    xaxis=dict(tickmode='linear'),  # Ensure all years are shown as ticks
    coloraxis_colorbar=dict(title='Return (%)'),  # Add a color bar for context
    template='plotly_white'
)

# Show the plot
fig2.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Load cumulative portfolio returns from Section 2
cumulative_portfolio_returns = pd.read_csv('/mnt/data/cumulative_returns.csv', index_col=0, parse_dates=True)

# Add an initial investment of ₹100,000
initial_investment = 100000
cumulative_portfolio_returns['Portfolio Value'] = cumulative_portfolio_returns['Portfolio Value'] * initial_investment

# Plot cumulative portfolio value as a stock-like line graph
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=cumulative_portfolio_returns.index,
    y=cumulative_portfolio_returns['Portfolio Value'],
    mode='lines',
    line=dict(color='blue', width=2),
    name='Portfolio Value'
))

# Add percentage labels at key intervals (year-end)
yearly_data = cumulative_portfolio_returns.resample('Y').last()
yearly_percentage = ((yearly_data['Portfolio Value'] / initial_investment) - 1) * 100

fig.add_trace(go.Scatter(
    x=yearly_data.index,
    y=yearly_data['Portfolio Value'],
    mode='markers+text',
    text=[f"{val:.2f}%" for val in yearly_percentage],
    textposition="top center",
    marker=dict(size=6, color='red'),
    name='Year-End Percentage Return'
))

# Update layout for readability
fig.update_layout(
    title='Cumulative Portfolio Returns (₹100,000 Initial Investment from 2010)',
    xaxis_title='Date',
    yaxis_title='Portfolio Value (₹)',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()


<ipython-input-6-38f4f7011be7>:23: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import plotly.express as px
import plotly.graph_objects as go

# Load regression results and stock data
regression_results = pd.read_csv('/mnt/data/regression_results.csv')  # Regression results from section 1
data = pd.read_csv('/content/combined_mid_large.csv')  # Stock data
data['Date'] = pd.to_datetime(data['Date'])

# Calculate daily returns for each stock
data['Daily_Return'] = data.groupby('symbol')['Close'].pct_change()
returns_df = data.pivot_table(values='Daily_Return', index='Date', columns='symbol').dropna()

# Log returns (for stability)
log_returns_df = np.log(1 + returns_df)
mean_returns = log_returns_df.mean() * 252  # Annualized
cov_matrix = log_returns_df.cov() * 252  # Annualized

# Risk-free rate
risk_free_rate = 0.03

# Portfolio Optimization Function
def portfolio_metrics(weights):
    portfolio_return = np.sum(weights * mean_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
    return portfolio_return, portfolio_volatility, sharpe_ratio

# Function to minimize negative Sharpe ratio
def negative_sharpe(weights):
    _, _, sharpe_ratio = portfolio_metrics(weights)
    return -sharpe_ratio

# Initial weights, bounds, and constraints
num_assets = len(mean_returns)
initial_weights = np.ones(num_assets) / num_assets
bounds = [(0, 1) for _ in range(num_assets)]
constraints = [{'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1}]

# Optimization
result = minimize(negative_sharpe, initial_weights, bounds=bounds, constraints=constraints)

# Optimal weights
optimal_weights = result.x
optimal_return, optimal_volatility, optimal_sharpe = portfolio_metrics(optimal_weights)

# Save best portfolio details
best_portfolio = pd.DataFrame({
    'Stock': mean_returns.index,
    'Weight': optimal_weights,
    'Expected Return': mean_returns.values,
    'Volatility': np.sqrt(np.diag(cov_matrix)),
})
best_portfolio['Risk'] = best_portfolio['Weight'] * best_portfolio['Volatility']

# Calculate portfolio returns over time (cumulative)
portfolio_returns = (returns_df * optimal_weights).sum(axis=1)
cumulative_portfolio_returns = (1 + portfolio_returns).cumprod()

# Calculate CAGR
start_value = 1  # Initial portfolio value (normalized to 1)
end_value = cumulative_portfolio_returns.iloc[-1]
n_years = (cumulative_portfolio_returns.index[-1] - cumulative_portfolio_returns.index[0]).days / 365.25
cagr = ((end_value / start_value) ** (1 / n_years)) - 1

# Save portfolio results
best_portfolio.to_csv('/mnt/data/best_portfolio.csv', index=False)
cumulative_portfolio_returns.to_csv('/mnt/data/cumulative_returns.csv', header=['Portfolio Value'])

# Print the CAGR
print(f"CAGR from 2010 to {cumulative_portfolio_returns.index[-1].year}: {cagr:.2%}")

# Visualization 1: Line Graph of Cumulative Portfolio Returns
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=cumulative_portfolio_returns.index,
    y=cumulative_portfolio_returns.values * 100000,  # Convert to ₹100,000 initial investment
    mode='lines',
    line=dict(color='blue', width=2),
    name='Portfolio Value'
))

# Add CAGR text at the final point of the graph
fig1.add_trace(go.Scatter(
    x=[cumulative_portfolio_returns.index[-1]],
    y=[cumulative_portfolio_returns.iloc[-1] * 100000],
    mode='text',
    text=[f"CAGR: {cagr:.2%}"],
    textposition="top right"
))

fig1.update_layout(
    title='Cumulative Portfolio Returns with ₹100,000 Initial Investment',
    xaxis_title='Date',
    yaxis_title='Portfolio Value (₹)',
    hovermode='x unified',
    template='plotly_white'
)

fig1.show()


CAGR from 2010 to 2024: 33.92%


In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import plotly.graph_objects as go

# Load regression results and stock data
data = pd.read_csv('/content/combined_mid_large.csv')  # Stock data
data['Date'] = pd.to_datetime(data['Date'])

# Calculate daily returns for each stock
data['Daily_Return'] = data.groupby('symbol')['Close'].pct_change()
returns_df = data.pivot_table(values='Daily_Return', index='Date', columns='symbol').dropna()

# Log returns (for stability)
log_returns_df = np.log(1 + returns_df)
mean_returns = log_returns_df.mean() * 252  # Annualized
cov_matrix = log_returns_df.cov() * 252  # Annualized

# Risk-free rate
risk_free_rate = 0.03

# Portfolio Optimization Function
def portfolio_metrics(weights):
    portfolio_return = np.sum(weights * mean_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
    return portfolio_return, portfolio_volatility, sharpe_ratio

# Function to minimize negative Sharpe ratio
def negative_sharpe(weights):
    _, _, sharpe_ratio = portfolio_metrics(weights)
    return -sharpe_ratio

# Initial weights, bounds, and constraints
num_assets = len(mean_returns)
initial_weights = np.ones(num_assets) / num_assets
bounds = [(0, 1) for _ in range(num_assets)]
constraints = [{'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1}]

# Optimization
result = minimize(negative_sharpe, initial_weights, bounds=bounds, constraints=constraints)

# Optimal weights
optimal_weights = result.x
optimal_return, optimal_volatility, optimal_sharpe = portfolio_metrics(optimal_weights)

# Calculate portfolio returns over time (cumulative)
portfolio_returns = (returns_df * optimal_weights).sum(axis=1)
cumulative_portfolio_returns = (1 + portfolio_returns).cumprod()

# Calculate CAGR
start_value = 1  # Initial portfolio value (normalized to 1)
end_value = cumulative_portfolio_returns.iloc[-1]
n_years = (cumulative_portfolio_returns.index[-1] - cumulative_portfolio_returns.index[0]).days / 365.25
cagr = ((end_value / start_value) ** (1 / n_years)) - 1

# Print the CAGR
print(f"CAGR from 2010 to {cumulative_portfolio_returns.index[-1].year}: {cagr:.2%}")

# Future Value Prediction for Next 10 Years
initial_investment = 100000
future_years = np.arange(1, 11)
future_values = initial_investment * (1 + cagr) ** future_years

# Visualization: Bar Graph of Predicted Portfolio Value for Next 10 Years
fig_future = go.Figure()

fig_future.add_trace(go.Bar(
    x=[f'Year {cumulative_portfolio_returns.index[-1].year + i}' for i in future_years],
    y=future_values,
    marker=dict(color='blue'),
    text=[f"₹{value:,.2f}" for value in future_values],
    textposition='outside'
))

fig_future.update_layout(
    title='Predicted Portfolio Value Over Next 10 Years (₹100,000 Investment)',
    xaxis_title='Year',
    yaxis_title='Portfolio Value (₹)',
    showlegend=False,
    template='plotly_white'
)

fig_future.show()


CAGR from 2010 to 2024: 33.92%


In [ ]:
import plotly.graph_objects as go
import numpy as np

# Future Value Prediction for Next 10 Years
initial_investment = 100000
future_years = np.arange(1, 11)
future_values = initial_investment * (1 + cagr) ** future_years

# Convert values to lakhs
future_values_lakhs = future_values / 100000

# Visualization: Bar Graph of Predicted Portfolio Value for Next 10 Years
fig_future = go.Figure()

fig_future.add_trace(go.Bar(
    x=[f'Year {cumulative_portfolio_returns.index[-1].year + i}' for i in future_years],
    y=future_values_lakhs,
    marker=dict(color='teal'),  # Changed color to teal
    text=[f"₹{value:,.2f} L" for value in future_values_lakhs],
    textposition='outside'
))

fig_future.update_layout(
    title='Predicted Portfolio Value Over Next 10 Years (₹100,000 Investment)',
    xaxis_title='Year',
    yaxis_title='Portfolio Value (₹ Lakhs)',
    showlegend=False,
    template='plotly_white',
    yaxis=dict(tickformat='.2f')
)

fig_future.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import plotly.express as px
import plotly.graph_objects as go

# Load regression results and stock data
regression_results = pd.read_csv('/mnt/data/regression_results.csv')  # Regression results from section 1
data = pd.read_csv('/content/combined_mid_large.csv')  # Stock data
data['Date'] = pd.to_datetime(data['Date'])

# Calculate daily returns for each stock
data['Daily_Return'] = data.groupby('symbol')['Close'].pct_change()
returns_df = data.pivot_table(values='Daily_Return', index='Date', columns='symbol').dropna()

# Log returns (for stability)
log_returns_df = np.log(1 + returns_df)
mean_returns = log_returns_df.mean() * 252  # Annualized
cov_matrix = log_returns_df.cov() * 252  # Annualized

# Risk-free rate
risk_free_rate = 0.03

# Portfolio Optimization Function
def portfolio_metrics(weights):
    portfolio_return = np.sum(weights * mean_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
    return portfolio_return, portfolio_volatility, sharpe_ratio

# Function to minimize negative Sharpe ratio
def negative_sharpe(weights):
    _, _, sharpe_ratio = portfolio_metrics(weights)
    return -sharpe_ratio

# Initial weights, bounds, and constraints
num_assets = len(mean_returns)
initial_weights = np.ones(num_assets) / num_assets
bounds = [(0, 1) for _ in range(num_assets)]
constraints = [{'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1}]

# Optimization
result = minimize(negative_sharpe, initial_weights, bounds=bounds, constraints=constraints)

# Optimal weights
optimal_weights = result.x
optimal_return, optimal_volatility, optimal_sharpe = portfolio_metrics(optimal_weights)

# Save best portfolio details
best_portfolio = pd.DataFrame({
    'Stock': mean_returns.index,
    'Weight': optimal_weights,
    'Expected Return': mean_returns.values,
    'Volatility': np.sqrt(np.diag(cov_matrix)),
})
best_portfolio['Risk'] = best_portfolio['Weight'] * best_portfolio['Volatility']
best_portfolio['Reward'] = best_portfolio['Weight'] * best_portfolio['Expected Return']

# Calculate portfolio returns over time (cumulative)
portfolio_returns = (returns_df * optimal_weights).sum(axis=1)
cumulative_portfolio_returns = (1 + portfolio_returns).cumprod()

# Save portfolio results
best_portfolio.to_csv('/mnt/data/best_portfolio.csv', index=False)
cumulative_portfolio_returns.to_csv('/mnt/data/cumulative_returns.csv', header=['Portfolio Value'])

# Save all portfolios tested
all_portfolios = []

# Grid search across weights for reporting purposes
for _ in range(500):  # Randomized search over portfolios
    random_weights = np.random.dirichlet(np.ones(num_assets))
    ret, vol, sharpe = portfolio_metrics(random_weights)
    all_portfolios.append({
        'Weights': random_weights,
        'Return': ret,
        'Volatility': vol,
        'Sharpe Ratio': sharpe
    })

all_portfolios_df = pd.DataFrame(all_portfolios)

# Extract individual stock weights and risks for all portfolios
all_portfolios_detailed = []
for portfolio in all_portfolios:
    weights = portfolio['Weights']
    risk_contributions = weights * np.sqrt(np.diag(cov_matrix))
    reward_contributions = weights * mean_returns
    all_portfolios_detailed.append({
        'Weights': weights,
        'Risk': risk_contributions.sum(),
        'Reward': reward_contributions.sum(),
        **{f'Weight_{stock}': weight for stock, weight in zip(mean_returns.index, weights)},
    })

all_portfolios_detailed_df = pd.DataFrame(all_portfolios_detailed)

# Save all portfolio details
all_portfolios_df.to_csv('/mnt/data/all_portfolios.csv', index=False)
all_portfolios_detailed_df.to_csv('/mnt/data/all_portfolios_detailed.csv', index=False)

# Visualization 1: Line Graph of Portfolio Value Over Time
fig1 = px.line(
    x=cumulative_portfolio_returns.index,
    y=cumulative_portfolio_returns.values,
    title='Cumulative Portfolio Returns (2010 - Present)',
    labels={'x': 'Date', 'y': 'Portfolio Value (₹)'}
)
fig1.show()

# Visualization 2: Bar Graph of Yearly Portfolio Returns
# (Note: Assumes 'annual_returns' DataFrame is available)
annual_returns = portfolio_returns.resample('Y').apply(lambda x: (1 + x).prod() - 1)
fig2 = px.bar(
    x=annual_returns.index.year,
    y=annual_returns.values * 100,
    title='Yearly Portfolio Returns (%) (2010 - Present)',
    labels={'x': 'Year', 'y': 'Annual Return (%)'}
)
fig2.update_layout(
    xaxis_title='Year',
    yaxis_title='Return (%)',
    xaxis=dict(tickmode='linear'),  # Ensure all years are shown as ticks
    template='plotly_white'
)
fig2.show()

# Visualization 3: Plot Showing Different Portfolios and Best One
fig3 = px.scatter(
    all_portfolios_df,
    x='Volatility',
    y='Return',
    color='Sharpe Ratio',
    title='Portfolios Tested (Volatility vs. Return)',
    labels={'Volatility': 'Risk (Volatility)', 'Return': 'Reward (Return)'},
    hover_data=['Sharpe Ratio']
)
fig3.add_trace(
    go.Scatter(
        x=[optimal_volatility],
        y=[optimal_return],
        mode='markers',
        marker=dict(color='red', size=12),
        name='Best Portfolio'
    )
)
fig3.show()

# Visualization 4: Bar Graph of Portfolio Weights
fig4 = px.bar(
    best_portfolio,
    x='Stock',
    y='Weight',
    title='Best Portfolio Weights',
    labels={'Weight': 'Allocation'},
    text='Weight'
)
fig4.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig4.show()


<ipython-input-3-9e633f552073>:113: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



### visualization section 3

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Load portfolio data from section 2
portfolios_df = pd.read_csv('/mnt/data/all_portfolios.csv')  # All portfolios tested
best_portfolio_df = pd.read_csv('/mnt/data/best_portfolio.csv')  # Best portfolio

# Load regression model performance data from section 1
regression_results_df = pd.read_csv('/mnt/data/regression_results.csv')

# Initial Capital
initial_capital = 100_000

# ======================= 1. Portfolio Performance Over Time ======================= #
# Calculate cumulative returns over time using the best portfolio weights
returns_df = pd.read_csv('/mnt/data/daily_returns.csv', index_col=0, parse_dates=True)
best_weights = best_portfolio_df['Weight'].values
cumulative_returns = ((returns_df * best_weights).sum(axis=1) + 1).cumprod() * initial_capital

fig1 = px.line(
    x=cumulative_returns.index,
    y=cumulative_returns.values,
    title='Portfolio Performance Over Time (Cumulative Returns)',
    labels={'x': 'Date', 'y': 'Portfolio Value (₹)'}
)
fig1.update_layout(xaxis_title='Date', yaxis_title='Portfolio Value (₹)')
fig1.show()

# ======================= 2. Annual Portfolio Returns ======================= #
# Calculate annual returns from cumulative returns
annual_returns = cumulative_returns.resample('Y').ffill().pct_change() * 100
fig2 = px.bar(
    x=annual_returns.index.year,
    y=annual_returns.values,
    title='Annual Portfolio Returns',
    labels={'x': 'Year', 'y': 'Return (%)'}
)
fig2.update_layout(xaxis_title='Year', yaxis_title='Return (%)')
fig2.show()

# ======================= 3. Risk vs. Return Scatter Plot ======================= #
fig3 = px.scatter(
    portfolios_df,
    x='Risk',
    y='Return',
    size='Sharpe Ratio',
    color='Sharpe Ratio',
    hover_data=['Weights'],
    title='Risk vs. Return (All Portfolios)',
)
# Highlight the best portfolio
fig3.add_trace(
    go.Scatter(
        x=[best_portfolio_df['Risk'].values[0]],
        y=[best_portfolio_df['Return'].values[0]],
        mode='markers',
        marker=dict(size=10, color='red', symbol='x'),
        name='Best Portfolio'
    )
)
fig3.update_layout(xaxis_title='Risk (Volatility)', yaxis_title='Return (%)')
fig3.show()

# ======================= 4. Stock Allocation in Optimized Portfolio ======================= #
fig4 = px.bar(
    best_portfolio_df,
    x='Stock',
    y='Weight',
    title='Stock Allocation in Optimized Portfolio',
    labels={'x': 'Stock', 'y': 'Weight (%)'},
    text='Weight'
)
fig4.show()

# ======================= 5. Feature Importance from Best Regression Model ======================= #
best_model = regression_results_df.sort_values('R2', ascending=False).iloc[0]['Model']
feature_importance_df = pd.read_csv(f'/mnt/data/{best_model}_feature_importance.csv')

fig5 = px.bar(
    feature_importance_df,
    x='Feature',
    y='Importance',
    title=f'Feature Importance ({best_model})',
    labels={'x': 'Feature', 'y': 'Importance Score'}
)
fig5.show()

# ======================= 6. Comparison of Regression Model Performance ======================= #
fig6 = px.bar(
    regression_results_df,
    x='Model',
    y='RMSE',
    title='Regression Model Comparison (RMSE)',
    labels={'x': 'Model', 'y': 'RMSE'}
)
fig6.show()

# ======================= 7. Portfolio Value Projection (Next 10 Years) ======================= #
avg_annual_return = cumulative_returns.pct_change().mean() * 252  # Average annual return
future_values = [initial_capital * ((1 + avg_annual_return) ** i) for i in range(1, 11)]
future_df = pd.DataFrame({'Year': list(range(2025, 2035)), 'Portfolio Value': future_values})

fig7 = px.line(
    future_df,
    x='Year',
    y='Portfolio Value',
    title='Portfolio Value Projection (Next 10 Years)',
    labels={'x': 'Year', 'y': 'Portfolio Value (₹)'}
)
fig7.update_traces(texttemplate='%{y:.2f}', textposition='outside')
fig7.show()

# ======================= 8. Portfolio Volatility Distribution ======================= #
fig8 = px.histogram(
    portfolios_df,
    x='Risk',
    nbins=10,
    title='Portfolio Volatility Distribution',
    labels={'x': 'Volatility', 'y': 'Frequency'}
)
fig8.update_layout(xaxis_title='Volatility', yaxis_title='Frequency')
fig8.show()

# ======================= Export Data & Visualizations ======================= #
# Export to CSVs
portfolios_df.to_csv('/mnt/data/all_tested_portfolios.csv', index=False)
best_portfolio_df.to_csv('/mnt/data/best_portfolio_results.csv', index=False)

print("All visualizations and data have been generated and exported.")


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/daily_returns.csv'

### downlord files

In [ ]:
from google.colab import files

# Make sure the cell that creates 'best_portfolio_results.csv' (ipython-input-18-ff099edaef87) has been executed before this line.
files.download('/mnt/data/all_portfolios_detailed.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>